# Human Activity Recognition — how much of the accuracy is real?
**CS6140 term project (Track 3) — starter notebook**

Dataset: UCI HAR (Anguita et al., 2013). 30 subjects, smartphone on the waist, 6 activities,
2.56 s windows (128 samples at 50 Hz) with 50% overlap, plus 561 precomputed features per window.

Experiments in this notebook:
1. **E1 Baseline** — classical models on the official subject-wise split, compared to the published SVM result.
2. **E2 Split ladder** — four splits that each remove one advantage, to separate overlap leakage, closeness in time, and person shift.
3. **E3 Deep model** — 1D-CNN on raw inertial signals vs. handcrafted features.
4. **E4 Error analysis** — confusion matrix, per-subject accuracy (leave-one-subject-out).
5. **E5 Scaling** — accuracy vs. number of training subjects (the "what happens with more data" question).

Run top to bottom on Colab (Runtime → Change runtime type → GPU helps for E3 only).

In [1]:
import os, zipfile, urllib.request, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold, LeaveOneGroupOut
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

SEED = 0
np.random.seed(SEED)

DATA_DIR = os.environ.get("HAR_DATA_DIR", "data")
ROOT = os.path.join(DATA_DIR, "UCI HAR Dataset")
OUT = "results"
os.makedirs(OUT, exist_ok=True)

CNN_EPOCHS = int(os.environ.get("CNN_EPOCHS", 30))
RUN_CNN = os.environ.get("RUN_CNN", "1") == "1"

## 1. Download and load the data

In [2]:
URLS = [
    "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
]

def _extract_all(zpath, dest):
    # The UCI download is a zip that contains another zip; unpack recursively.
    with zipfile.ZipFile(zpath) as z:
        z.extractall(dest)
        inner = [n for n in z.namelist() if n.endswith(".zip") and not n.startswith("__MACOSX")]
    for n in inner:
        _extract_all(os.path.join(dest, n), dest)

def download():
    if os.path.isdir(ROOT):
        print("Data already present at", ROOT)
        return
    os.makedirs(DATA_DIR, exist_ok=True)
    zpath = os.path.join(DATA_DIR, "har.zip")
    for url in URLS:
        try:
            print("Downloading", url)
            urllib.request.urlretrieve(url, zpath)
            break
        except Exception as e:
            print("  failed:", e)
    else:
        raise RuntimeError("Download failed. Download the zip manually from the UCI repository "
                           "and unzip it so that '" + ROOT + "' exists.")
    _extract_all(zpath, DATA_DIR)
    assert os.path.isdir(ROOT), "Unzipped, but '" + ROOT + "' not found; check folder names."

download()

Data already present at data/UCI HAR Dataset


In [3]:
SIGNALS = [f"{kind}_{axis}" for kind in ("body_acc", "body_gyro", "total_acc") for axis in "xyz"]
ACTIVITIES = ["WALKING", "UPSTAIRS", "DOWNSTAIRS", "SITTING", "STANDING", "LAYING"]

def _read(path, dtype=float):
    return pd.read_csv(path, sep=r"\s+", header=None).to_numpy(dtype=dtype)

def load_split(split):
    d = os.path.join(ROOT, split)
    X = _read(os.path.join(d, f"X_{split}.txt"))                        # (n, 561) handcrafted features
    y = _read(os.path.join(d, f"y_{split}.txt"), int).ravel() - 1       # labels 0..5
    s = _read(os.path.join(d, f"subject_{split}.txt"), int).ravel()     # subject ids 1..30
    raw = np.stack([_read(os.path.join(d, "Inertial Signals", f"{name}_{split}.txt"))
                    for name in SIGNALS], axis=1).astype(np.float32)    # (n, 9, 128) raw signals
    return X, y, s, raw

Xtr, ytr, str_, Rtr = load_split("train")
Xte, yte, ste, Rte = load_split("test")

# Pooled copy (all 30 subjects) for the cross-validation experiments
Xall = np.vstack([Xtr, Xte]); yall = np.concatenate([ytr, yte])
sall = np.concatenate([str_, ste]); Rall = np.concatenate([Rtr, Rte])

### Data summary (this is the evidence for the proposal that the data is in hand)

In [4]:
print("Train windows:", Xtr.shape, "| raw:", Rtr.shape, "| subjects:", sorted(map(int, set(str_))))
print("Test  windows:", Xte.shape, "| raw:", Rte.shape, "| subjects:", sorted(map(int, set(ste))))
print("Subject overlap between official train and test:", set(str_) & set(ste) or "none")
print(pd.Series(yall).map(dict(enumerate(ACTIVITIES))).value_counts().sort_index())

Train windows: (7352, 561) | raw: (7352, 9, 128) | subjects: [1, 3, 5, 6, 7, 8, 11, 14, 15, 16, 17, 19, 21, 22, 23, 25, 26, 27, 28, 29, 30]
Test  windows: (2947, 561) | raw: (2947, 9, 128) | subjects: [2, 4, 9, 10, 12, 13, 18, 20, 24]
Subject overlap between official train and test: none
DOWNSTAIRS    1406
LAYING        1944
SITTING       1777
STANDING      1906
UPSTAIRS      1544
WALKING       1722
Name: count, dtype: int64


### Overlap check and recording segments
If consecutive rows come from one continuous recording, the second half of window *i* (samples 64–127) should be
exactly the first half of window *i+1* (samples 0–63). This cell checks that, then groups the windows into
**segments**: unbroken runs of one person doing one activity. Segments are needed for the time-based split in E2.

In [5]:
def find_segments(R, y, s, tol=1e-6):
    match = np.isclose(R[:-1, :, 64:], R[1:, :, :64], atol=tol).all(axis=(1, 2))   # halves identical?
    same = (s[:-1] == s[1:]) & (y[:-1] == y[1:])                                  # same person + activity?
    print(f"  consecutive pairs that overlap exactly: {match.mean():.4f}")
    print(f"  ...among pairs with same person + activity: {match[same].mean():.4f}")
    if match[same].mean() < 0.9:
        print("  WARNING: overlap is not exact. Falling back to person+activity runs as segments.")
        cont = same
    else:
        cont = match & same
    return np.concatenate([[0], np.cumsum(~cont)])   # new segment id wherever the chain breaks

print("Train:"); seg_tr = find_segments(Rtr, ytr, str_)
print("Test:");  seg_te = find_segments(Rte, yte, ste)
seg_all = np.concatenate([seg_tr, seg_te + seg_tr.max() + 1])
pos_all = pd.Series(seg_all).groupby(seg_all).cumcount().to_numpy()   # position of each window inside its segment
seg_len = pd.Series(seg_all).map(pd.Series(seg_all).value_counts()).to_numpy()
lens = pd.Series(seg_all).value_counts()
print(f"Segments: {len(lens)} | windows per segment: median {lens.median():.0f}, min {lens.min()}, max {lens.max()}")

Train:
  consecutive pairs that overlap exactly: 0.9185
  ...among pairs with same person + activity: 0.9548
Test:
  consecutive pairs that overlap exactly: 0.9131
  ...among pairs with same person + activity: 0.9515
Segments: 857 | windows per segment: median 12, min 1, max 29


In [6]:
segs = pd.DataFrame({"seg": seg_all, "person": sall, "act": yall}).groupby("seg").agg(
    person=("person", "first"), act=("act", "first"), n=("seg", "size"))
print("Segments per person per activity:")
print(segs.groupby(["person", "act"]).size().describe().round(1))
print("Windows in segments shorter than 5:", segs.loc[segs.n < 5, "n"].sum(), "of", len(seg_all))

Segments per person per activity:
count    180.0
mean       4.8
std        1.1
min        4.0
25%        4.0
50%        4.0
75%        6.0
max        8.0
dtype: float64
Windows in segments shorter than 5: 37 of 10299


## 2. E1 — Baseline on the official subject-wise split
External ground truth: Anguita et al. (2013) report roughly 96% test accuracy with a multiclass SVM on these
561 features. Check the exact number in the paper and cite it in the report.

In [7]:
def make_models():
    return {
        "LogReg":   make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
        "SVM-RBF":  make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, gamma="scale")),
        "RandomForest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED),
        "HistGB":   HistGradientBoostingClassifier(random_state=SEED),
    }

def score(y_true, y_pred):
    return {"acc": accuracy_score(y_true, y_pred), "macro_f1": f1_score(y_true, y_pred, average="macro")}

In [8]:
e1 = {}
fitted = {}
for name, m in make_models().items():
    t = time.time()
    m.fit(Xtr, ytr)
    pred = m.predict(Xte)
    e1[name] = {**score(yte, pred), "fit_sec": round(time.time() - t, 1)}
    fitted[name] = m
    print(f"{name:13s} acc={e1[name]['acc']:.4f}  macroF1={e1[name]['macro_f1']:.4f}  ({e1[name]['fit_sec']}s)")

e1_df = pd.DataFrame(e1).T
e1_df.to_csv(f"{OUT}/e1_official_split.csv")
e1_df

LogReg        acc=0.9549  macroF1=0.9547  (0.2s)
SVM-RBF       acc=0.9549  macroF1=0.9540  (2.5s)
RandomForest  acc=0.9253  macroF1=0.9232  (3.2s)
HistGB        acc=0.9335  macroF1=0.9329  (7.8s)


,acc,macro_f1,fit_sec
LogReg,0.954869,0.954675,0.2
SVM-RBF,0.954869,0.953953,2.5
RandomForest,0.925348,0.923168,3.2
HistGB,0.933492,0.932949,7.8


### Tuning with person-wise cross-validation
Hyperparameters are chosen with GroupKFold on the 21 training people only. The official test people are used once, at the end.

In [10]:
from sklearn.model_selection import GridSearchCV
grids = {
    "LogReg": (make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
               {"logisticregression__C": [0.01, 0.1, 1, 10]}),
    "SVM-RBF": (make_pipeline(StandardScaler(), SVC()),
                {"svc__C": [10, 100, 1000], "svc__gamma": [1e-3, 1e-4, 1e-5]}),
}
tuned = {}
for name, (model, grid) in grids.items():
    gs = GridSearchCV(model, grid, cv=GroupKFold(5), scoring="accuracy", n_jobs=-1)
    gs.fit(Xtr, ytr, groups=str_)
    pred = gs.predict(Xte)
    tuned[name] = {"best_params": gs.best_params_, "cv_acc": gs.best_score_, **score(yte, pred)}
    print(f"{name:8s} best {gs.best_params_} | person-wise CV acc {gs.best_score_:.4f} | test acc {tuned[name]['acc']:.4f}")
pd.DataFrame(tuned).T.to_csv(f"{OUT}/e1_tuned.csv")

LogReg   best {'logisticregression__C': 0.1} | person-wise CV acc 0.9348 | test acc 0.9498
SVM-RBF  best {'svc__C': 100, 'svc__gamma': 0.0001} | person-wise CV acc 0.9332 | test acc 0.9562


## 3. E2 — Split ladder: what part of the gap is leakage, and what part is new people?
Comparing a random split with a person split changes two things at once. These four splits change one thing at a time:

| Split | Overlapping neighbours in train? | Nearby (non-overlapping) windows in train? | Same person in train? |
|---|---|---|---|
| A. Random, all windows | yes | yes | yes |
| B. Random, non-overlapping windows only | no | yes | yes |
| C. Same people, split by time inside each segment (+ gap) | no | no | yes |
| D. Split by person (GroupKFold) | no | no | no |

Gaps: **A − B = overlap leakage**, **B − C = closeness in time**, **C − D = person shift**.
To keep the comparison fair, every split trains on the same number of windows.

In [12]:
def ladder_folds(n_splits=5, gap=1, match_size=True):
    rng = np.random.default_rng(SEED)
    idx_all = np.arange(len(yall))
    keep = pos_all % 2 == 0                      # every other window inside a segment -> no shared samples
    idx_sub = idx_all[keep]
    skf = StratifiedKFold(n_splits, shuffle=True, random_state=SEED)
    folds = {}
    folds["A random, all windows"] = [(idx_all[a], idx_all[b]) for a, b in skf.split(idx_all, yall)]
    folds["B random, non-overlapping"] = [(idx_sub[a], idx_sub[b]) for a, b in skf.split(idx_sub, yall[idx_sub])]

    # C: inside every segment, cut the kept windows into n_splits blocks in time order
    df = pd.DataFrame({"i": idx_sub, "seg": seg_all[idx_sub], "pos": pos_all[idx_sub] // 2})
    df["len"] = df.groupby("seg")["pos"].transform("count")
    df["block"] = np.floor(df["pos"] / df["len"] * n_splits).astype(int)
    C_folds = []
    for k in range(n_splits):
        te = df[df.block == k]
        lo = te.groupby("seg")["pos"].min(); hi = te.groupby("seg")["pos"].max()
        tr = df[df.block != k]
        near = (tr["pos"] >= tr["seg"].map(lo) - gap) & (tr["pos"] <= tr["seg"].map(hi) + gap)   # NaN -> False
        C_folds.append((tr[~near]["i"].to_numpy(), te["i"].to_numpy()))
    folds["C same people, split by time"] = C_folds

    folds["D split by person"] = [(idx_sub[a], idx_sub[b]) for a, b in GroupKFold(n_splits).split(idx_sub, groups=sall[idx_sub])]

    n_train = min(len(tr) for f in folds.values() for tr, _ in f)   # same training size everywhere
    if match_size:
        folds = {name: [(rng.choice(tr, n_train, replace=False), te) for tr, te in f] for name, f in folds.items()}
    return folds, n_train

ladder, n_train = ladder_folds()
print("Training windows per fold (all splits):", n_train)

ladder_models = {
    "LogReg": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    "RandomForest": lambda: RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED),
}
rows = []
for split_name, folds in ladder.items():
    for model_name, make in ladder_models.items():
        accs, f1s = [], []
        for tr, te in folds:
            m = make().fit(Xall[tr], yall[tr]); p = m.predict(Xall[te])
            accs.append(accuracy_score(yall[te], p)); f1s.append(f1_score(yall[te], p, average="macro"))
        rows.append({"split": split_name, "model": model_name, "acc_mean": np.mean(accs),
                     "acc_std": np.std(accs), "macro_f1": np.mean(f1s)})
        print(f"{split_name:30s} {model_name:13s} acc {np.mean(accs):.4f} ± {np.std(accs):.4f}")
e2_df = pd.DataFrame(rows)
e2_df.to_csv(f"{OUT}/e2_split_ladder.csv", index=False)

acc = e2_df.pivot(index="model", columns="split", values="acc_mean")
A, B, Cs, D = acc.columns   # columns sort as A, B, C, D
gaps = pd.DataFrame({"total (A-D)": acc[A] - acc[D], "overlap leakage (A-B)": acc[A] - acc[B],
                     "closeness in time (B-C)": acc[B] - acc[Cs], "person shift (C-D)": acc[Cs] - acc[D]})
gaps.to_csv(f"{OUT}/e2_gaps.csv")
display(acc.round(4)); display((gaps * 100).round(2))   # gaps in percentage points

Training windows per fold (all splits): 2570
A random, all windows          LogReg        acc 0.9693 ± 0.0042
A random, all windows          RandomForest  acc 0.9616 ± 0.0022
B random, non-overlapping      LogReg        acc 0.9694 ± 0.0037
B random, non-overlapping      RandomForest  acc 0.9578 ± 0.0036
C same people, split by time   LogReg        acc 0.9462 ± 0.0185
C same people, split by time   RandomForest  acc 0.9358 ± 0.0229
D split by person              LogReg        acc 0.9389 ± 0.0181
D split by person              RandomForest  acc 0.9288 ± 0.0261


split,"A random, all windows","B random, non-overlapping","C same people, split by time",D split by person
model,,,,
LogReg,0.9693,0.9694,0.9462,0.9389
RandomForest,0.9616,0.9578,0.9358,0.9288


,total (A-D),overlap leakage (A-B),closeness in time (B-C),person shift (C-D)
model,,,,
LogReg,3.05,-0.00,2.31,0.74
RandomForest,3.29,0.39,2.20,0.70


In [ ]:
# Same ladder, but every split keeps its full training set
def run_ladder(folds_dict):
    rows = []
    for split_name, folds in folds_dict.items():
        for model_name, make in ladder_models.items():
            accs = [accuracy_score(yall[te], make().fit(Xall[tr], yall[tr]).predict(Xall[te])) for tr, te in folds]
            rows.append({"split": split_name, "model": model_name, "acc": np.mean(accs),
                         "n_train": int(np.mean([len(tr) for tr, _ in folds]))})
    return pd.DataFrame(rows)

full = run_ladder(ladder_folds(match_size=False)[0])
full.to_csv(f"{OUT}/e2_split_ladder_full_train.csv", index=False)
display(full.pivot(index="model", columns="split", values="acc").round(4))
display(full.groupby("split")["n_train"].first())

### Can a model tell *who* is moving?
If the 561 features identify the person well above chance (1/30 ≈ 3.3%), each person has a strong "signature",
which helps explain why new people are harder.

In [ ]:
idx_sub = np.arange(len(yall))[pos_all % 2 == 0]
pid = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
accs = []
for a, b in StratifiedKFold(5, shuffle=True, random_state=SEED).split(idx_sub, sall[idx_sub]):
    tr, te = idx_sub[a], idx_sub[b]
    accs.append(accuracy_score(sall[te], pid.fit(Xall[tr], sall[tr]).predict(Xall[te])))
print(f"Person-ID accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f} (chance {1/30:.4f})")

## 4. E3 — 1D-CNN on raw signals
Learns features from the 9×128 raw signals instead of using the 561 handcrafted ones.
Normalization statistics come from the training subjects only.

In [ ]:
if RUN_CNN:
    import torch, torch.nn as nn
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(SEED)

    class HARCNN(nn.Module):
        def __init__(self, n_ch=9, n_cls=6):
            super().__init__()
            def block(i, o, k): return nn.Sequential(nn.Conv1d(i, o, k, padding=k // 2), nn.BatchNorm1d(o), nn.ReLU())
            self.net = nn.Sequential(block(n_ch, 64, 5), block(64, 64, 5), nn.MaxPool1d(2),
                                     block(64, 128, 3), nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                     nn.Dropout(0.3), nn.Linear(128, n_cls))
        def forward(self, x): return self.net(x)

    def train_cnn(Rtrain, ytrain, Rtest, ytest, epochs=CNN_EPOCHS, bs=64, lr=1e-3):
        mu = Rtrain.mean(axis=(0, 2), keepdims=True); sd = Rtrain.std(axis=(0, 2), keepdims=True) + 1e-8
        to_t = lambda a: torch.tensor((a - mu) / sd, dtype=torch.float32)
        Xa, Xb = to_t(Rtrain), to_t(Rtest)
        ya = torch.tensor(ytrain, dtype=torch.long)
        model = HARCNN().to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        lossf = nn.CrossEntropyLoss()
        history = []
        for ep in range(epochs):
            model.train()
            perm = torch.randperm(len(Xa))
            for i in range(0, len(Xa), bs):
                idx = perm[i:i + bs]
                opt.zero_grad()
                loss = lossf(model(Xa[idx].to(device)), ya[idx].to(device))
                loss.backward(); opt.step()
            model.eval()
            with torch.no_grad():
                pred = torch.cat([model(Xb[i:i + 512].to(device)).argmax(1).cpu() for i in range(0, len(Xb), 512)]).numpy()
            history.append(accuracy_score(ytest, pred))
            if (ep + 1) % 5 == 0 or ep == epochs - 1:
                print(f"epoch {ep + 1:3d}  test acc {history[-1]:.4f}")
        return pred, history

    cnn_pred, cnn_hist = train_cnn(Rtr, ytr, Rte, yte)
    e1_df.loc["1D-CNN (raw)"] = {**score(yte, cnn_pred), "fit_sec": np.nan}
    e1_df.to_csv(f"{OUT}/e1_official_split.csv")
    # Note: watching test accuracy every epoch is for monitoring only. For the report, pick the epoch count
    # with a validation split of held-out *training subjects*, not the test set.
e1_df

## 5. E4 — Error analysis

In [ ]:
best = e1_df["acc"].drop(labels=["1D-CNN (raw)"], errors="ignore").idxmax()
pred = fitted[best].predict(Xte)
cm = confusion_matrix(yte, pred, normalize="true")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(6), ACTIVITIES, rotation=45, ha="right"); ax.set_yticks(range(6), ACTIVITIES)
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", color="white" if cm[i, j] > 0.5 else "black", fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"Confusion matrix — {best} (official test)")
plt.colorbar(im); plt.tight_layout(); plt.savefig(f"{OUT}/e4_confusion.png", dpi=150); plt.show()

print(classification_report(yte, pred, target_names=ACTIVITIES, digits=3))

In [ ]:
# Leave-one-person-out: how much does accuracy change from person to person?
loso_models = {
    "LogReg": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    "RandomForest": lambda: RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED),
}
per_subject = {name: {} for name in loso_models}
for tr, te in LeaveOneGroupOut().split(Xall, yall, sall):
    for name, make in loso_models.items():
        per_subject[name][int(sall[te][0])] = accuracy_score(yall[te], make().fit(Xall[tr], yall[tr]).predict(Xall[te]))
ps = pd.DataFrame(per_subject)
ps.to_csv(f"{OUT}/e4_loso_per_subject.csv")
summary = pd.DataFrame({"mean": ps.mean(), "std": ps.std(), "worst": ps.min(),
                        "worst_person": ps.idxmin(), "best": ps.max()})
display(summary.round(4))

order = ps["LogReg"].sort_values().index
ax = ps.loc[order].plot.bar(figsize=(10, 3.2), width=0.8)
ax.set_xlabel("person (sorted by LogReg accuracy)"); ax.set_ylabel("accuracy")
ax.set_ylim(max(0, ps.values.min() - 0.05), 1.0); ax.set_title("Leave-one-person-out accuracy")
plt.tight_layout(); plt.savefig(f"{OUT}/e4_loso.png", dpi=150); plt.show()

## 6. E5 — Accuracy vs. number of training subjects
Train on k randomly chosen training subjects, always test on the 9 official test subjects. Repeat with several
random subject draws so the error bars reflect *which* people you happened to collect.

In [ ]:
train_subjects = np.array(sorted(map(int, set(str_))))
ks = [k for k in (2, 4, 8, 12, 16, len(train_subjects)) if k <= len(train_subjects)]
rng = np.random.default_rng(SEED)
rows = []
for k in ks:
    for rep in range(10 if k < len(train_subjects) else 1):
        chosen = rng.choice(train_subjects, size=k, replace=False)
        mask = np.isin(str_, chosen)
        for name in ("LogReg", "RandomForest"):
            m = make_models()[name]
            m.fit(Xtr[mask], ytr[mask])
            rows.append({"k": k, "rep": rep, "model": name, "acc": accuracy_score(yte, m.predict(Xte))})
e5_df = pd.DataFrame(rows)
e5_df.to_csv(f"{OUT}/e5_learning_curve.csv", index=False)

agg = e5_df.groupby(["model", "k"])["acc"].agg(["mean", "std"]).reset_index()
plt.figure(figsize=(6, 4))
for name, g in agg.groupby("model"):
    plt.errorbar(g["k"], g["mean"], yerr=g["std"].fillna(0), marker="o", capsize=3, label=name)
plt.xlabel("number of training subjects"); plt.ylabel("test accuracy (unseen subjects)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.savefig(f"{OUT}/e5_learning_curve.png", dpi=150); plt.show()
agg

## Next steps (after the checkpoint)
- 1D-CNN under the split ladder too (does a deep model exploit leakage more or less than RF?)
- Optional: fine-tune a pretrained self-supervised model (Yuan et al., 2024) and see if pretraining shrinks the person-shift gap
- Confusion analysis per person: which people fail, and on which activities?